# Silver Layer: Data Cleaning and Validation

import pandas as pd
import numpy as np
import string
import datetime

### Load Bronze Dataset
Load the Bronze-level Parquet file to begin the Silver-layer transformation.
We first confirm the dataset size and structure to ensure successful ingestion.

In [1]:
import importlib
import pandas as pd


parquet_engine = None
for candidate in ('pyarrow', 'fastparquet'):
    if importlib.util.find_spec(candidate):
        parquet_engine = candidate
        break

if parquet_engine is None:
    raise ImportError(
        "Parquet support is missing. Install `pyarrow` or `fastparquet` in this notebook environment, then restart the kernel."
    )

print(f'Parquet engine available: {parquet_engine}')


Parquet engine available: pyarrow


In [2]:
df = pd.read_parquet('./Medallion Architecture/bronze/bronze_transactions.parquet')
print("Bronze dataset loaded:", df.shape)

Bronze dataset loaded: (48000, 21)


### Define Data Cleaning Helper Functions
Below are reusable functions to standardize and validate data across multiple cleaning steps.

- **drop_null_columns:** identifies and removes columns with a high null ratio (not used, but defined for reference).
- **fill_missing_values:** replaces NaNs with logical defaults.
- **convert_column_types:** converts columns to target datatypes (e.g., string, bool, int32).
- **remove_punctuation:** strips unwanted characters from text columns.
- **check_formats:** validates column types against expected schema.
- **check_similarity / check_misspelling:** detects near-duplicate or misspelled player names.


In [3]:
def drop_null_columns(df, threshold=0.2):
    null_percentage = df.isnull().mean()
    columns_to_keep = null_percentage[null_percentage < threshold].index
    if len(columns_to_keep) == len(df.columns):
        print('No columns removed.')
    else:
        print(f'removing:{[c for c in df.columns if c not in columns_to_keep]}')
    return df[columns_to_keep]

In [4]:
def fill_missing_values(df, columns_default: dict):
    
    for column, default_value in columns_default.items():
        if column in df.columns:
            df[column] = df[column].fillna(default_value)
    return df

In [5]:
def convert_column_types(df, columns_types: dict):
    for column, dtype in columns_types.items():
        if column not in df.columns:
            continue

        try:
            if dtype == 'bool':
                normalized = df[column].astype('string').str.strip().str.title()
                df[column] = normalized.map({'Yes': True, 'No': False}).astype('boolean')
            elif dtype.startswith('int'):
                df[column] = pd.to_numeric(df[column], errors='coerce').fillna(0).round().astype(dtype)
            elif dtype.startswith('float'):
                df[column] = pd.to_numeric(df[column], errors='coerce').astype(dtype)
            elif dtype == 'string':
                df[column] = df[column].astype('string')
            else:
                df[column] = df[column].astype(dtype)
        except Exception as e:
            print(f"Column '{column}' caused an issue: {e}")
    return df


In [6]:
def remove_punctuation(df, columns: list):
    for c in columns:
        if c in df.columns:
            df.loc[:, c] = df[c].astype('string').str.replace(r'[^\w\s]|_', '', regex=True)
    return df


In [7]:
def check_formats(df, expected_formats: dict):
    incorrect_formats = []

    for column, datatype in df.dtypes.to_dict().items():
        expected_type = expected_formats.get(column)
        if expected_type != str(datatype):
            incorrect_formats.append((column, str(datatype), expected_type))

    incorrect_columns = [c[0] for c in incorrect_formats]
    correct_format_count = len([c for c in df.columns if c not in incorrect_columns])

    if incorrect_formats:
        print('Below are incorrect formats')
        print('-' * 50)
        print(f'Correct column count: {correct_format_count}')
        return pd.DataFrame(incorrect_formats, columns=['Column', 'Actual', 'Expected'])
    else:
        print('Validation complete — no discrepancies.')


In [8]:
def check_similarity(word1: str, word2: str) -> float:
    word_set1 = set(word1)
    word_set2 = set(word2)

    intersection = word_set1.intersection(word_set2)
    intersection_count = len(intersection)

    total_char_count = len(word_set1.union(word_set2))

    similarity = intersection_count / total_char_count
    return similarity

In [9]:
def check_misspelling(dataframe: pd.DataFrame, column: str, similarity_threshold: float) -> pd.DataFrame:
    all_unique_values = list(set(dataframe[column].dropna().tolist()))

    similarity_list = []

    for n in range(len(all_unique_values)):
        value1 = all_unique_values[n]
        for n2 in range(n + 1, len(all_unique_values)):
            value2 = all_unique_values[n2]
            similarity = round(check_similarity(value1, value2), 4)
            if similarity >= similarity_threshold:
                similarity_list.append([value1, value2, similarity])

    return pd.DataFrame(similarity_list, columns=['name1', 'name2', 'similarity'])


### Inspect Initial Dataset
Display the first few rows and count total records and columns.
This helps establish a baseline before transformations.

In [10]:
df.head()

,Player Name,Team,World,Vehicle Type,Companion,Kart Racing Rank,Platforming Rank,Boss Battle Rank,Power-Ups Used,Kart Role,...,Lives Lost,Participation in Battle Mode,Mushroom Cup Participation,Power-Ups Owned,Coins Spent in Toad Town,Levels Completed,Times Hit by Enemies,Primary Game,fileName,loadDatetimeStamp
0,Yoshi,Toad Brigade,Yoshi's Island,Comet Bike,pOLTERPUP,A,A,A,12,Drifter,...,0.0,Yes,No,1-Up Mushroom,64,26,4.26,Mario Tennis Aces,mario_data_20250101.csv,2026-03-18 11:24:40.640227
1,PeachK,GREEN CAPS,Donut Plains,Circuit Special,kOOPA tROOPA,C,A,B,16,Drifter,...,4.0,No,No,"Red Shell, Super Star",335,40,5.00,Mario Tennis Aces,mario_data_20250101.csv,2026-03-18 11:24:40.640227
2,Waluigi,None,Yoshi's Island,Biddybuggy,Goomba,D,A,C,26,Blocker,...,1.0,No,Yes,Green Shell,182,57,5.50,Mario Kart 8 Deluxe,mario_data_20250101.csv,2026-03-18 11:24:40.640227
3,Yoshi,Toad Brigade,Star World,Pipe Frame,Goomba,C,D,A,23,Drifter,...,5.0,No,Yes,1-Up Mushroom,333,84,6.00,Super Mario Bros.,mario_data_20250101.csv,2026-03-18 11:24:40.640227
4,Bowser Jr.,Koopa Clan,Mushroom Kingdom,Pipe Frame,tOAD,C,C,B,10,Blocker,...,2.0,Yes,No,"Red Shell, Banana Peel, Fire Flower",461,55,7.00,Super Mario World,mario_data_20250101.csv,2026-03-18 11:24:40.640227


In [11]:
df.shape

(48000, 21)

### Check for Missing Values
Use `.isnull().sum()` and `.isnull().mean()` to identify how many nulls exist per column and what percentage of total records they represent.
This step informs which fields require imputation.

In [12]:
df.isnull().sum()

Player Name                      9040
Team                             2960
World                               0
Vehicle Type                        0
Companion                           0
Kart Racing Rank                 9280
Platforming Rank                    0
Boss Battle Rank                    0
Power-Ups Used                      0
Kart Role                           0
Team Points                         0
Lives Lost                          0
Participation in Battle Mode        0
Mushroom Cup Participation      12400
Power-Ups Owned                     0
Coins Spent in Toad Town            0
Levels Completed                    0
Times Hit by Enemies                0
Primary Game                        0
fileName                            0
loadDatetimeStamp                   0
dtype: int64

In [13]:
df.isnull().mean() * 100

Player Name                     18.833333
Team                             6.166667
World                            0.000000
Vehicle Type                     0.000000
Companion                        0.000000
Kart Racing Rank                19.333333
Platforming Rank                 0.000000
Boss Battle Rank                 0.000000
Power-Ups Used                   0.000000
Kart Role                        0.000000
Team Points                      0.000000
Lives Lost                       0.000000
Participation in Battle Mode     0.000000
Mushroom Cup Participation      25.833333
Power-Ups Owned                  0.000000
Coins Spent in Toad Town         0.000000
Levels Completed                 0.000000
Times Hit by Enemies             0.000000
Primary Game                     0.000000
fileName                         0.000000
loadDatetimeStamp                0.000000
dtype: float64

### Fill Missing Values
Replace missing values in categorical and boolean columns with appropriate defaults to preserve record integrity.
For example:
- `'Player Name' → 'Unknown Player'`
- `'Team' → 'No Team'`
- `'Kart Racing Rank' → 'Unranked'`
- `'Mushroom Cup Participation' → False`

In [14]:
df = fill_missing_values(
    df,
    {
        'Player Name': 'Unknown Player',
        'Team': 'No Team',
        'Kart Racing Rank': 'Unranked',
        'Mushroom Cup Participation': 'No',
        'Participation in Battle Mode': 'No'
    }
)


In [15]:
df.isnull().mean() * 100

Player Name                     0.0
Team                            0.0
World                           0.0
Vehicle Type                    0.0
Companion                       0.0
Kart Racing Rank                0.0
Platforming Rank                0.0
Boss Battle Rank                0.0
Power-Ups Used                  0.0
Kart Role                       0.0
Team Points                     0.0
Lives Lost                      0.0
Participation in Battle Mode    0.0
Mushroom Cup Participation      0.0
Power-Ups Owned                 0.0
Coins Spent in Toad Town        0.0
Levels Completed                0.0
Times Hit by Enemies            0.0
Primary Game                    0.0
fileName                        0.0
loadDatetimeStamp               0.0
dtype: float64

### Remove Punctuation from Text Columns
Clean up inconsistent punctuation and symbols in text-based fields like `'Kart Role'` to ensure standardized grouping.


In [16]:
df = remove_punctuation(df, ['Kart Role'])

In [17]:
df['Kart Role'].unique()[:10]

array(['Drifter', 'Blocker', 'Speedster', 'Item Specialist', 'Driver'],
      dtype=object)

### Detect Similar or Misspelled Player Names
Use the helper function `check_misspelling()` to identify pairs of names with ≥80% similarity.
This helps find variants like “Mariqo” or “Pjeach”.


In [18]:
similar_names = check_misspelling(df, 'Player Name', 0.8)
print('Step 3 complete - similar Player names (>= 80% similarity):')
display(similar_names)

Step 3 complete - similar Player names (>= 80% similarity):


,name1,name2,similarity
0,Yaoshi,YoshiY,0.8333
1,Yaoshi,YoshYi,0.8333
2,Yaoshi,Yioshi,0.8333
3,Yaoshi,Yosihi,0.8333
4,Yaoshi,Yohshi,0.8333
...,...,...,...
21346,Bowser Jr.v,Bowser JrI.,0.8182
21347,Walduigi,WaWluigi,0.8571
21348,Rosaliona,RosaGlina,0.8750
21349,Walvuigi,WaWluigi,0.8571


### Validate Expected Data Types
Compare each column’s current data type against the expected schema to catch mismatches before conversion.

In [19]:
expected_formats = {
    'Player Name': 'string',
    'Team': 'string',
    'Kart Racing Rank': 'string',
    'Mushroom Cup Participation': 'bool',
    'Participation in Battle Mode': 'bool',
    'Platforming Rank': 'string',
    'Boss Battle Rank': 'string',
    'Power-Ups Owned': 'string',
    'Power-Ups Used': 'int32',
    'Team Points': 'int32',
    'Levels Completed': 'int32',
    'Lives Lost': 'int32',
    'Times Hit by Enemies': 'int32',
    'Vehicle Type': 'string',
    'World': 'string',
    'Coins Spent in Toad Town': 'int32',
    'Companion': 'string',
    'Primary Game': 'string',
    'Kart Role': 'string'
}


In [20]:
check_formats(df, expected_formats=expected_formats)

Below are incorrect formats
--------------------------------------------------
Correct column count: 0


,Column,Actual,Expected
0,Player Name,object,string
1,Team,object,string
2,World,object,string
3,Vehicle Type,object,string
4,Companion,object,string
5,Kart Racing Rank,object,string
6,Platforming Rank,object,string
7,Boss Battle Rank,object,string
8,Power-Ups Used,int64,int32
9,Kart Role,object,string


### Convert Columns to Correct Data Types
Convert numeric fields to `int32` and categorical fields to `string`.  
Before conversion, fill any remaining NaNs in numeric columns with `0` to avoid errors.

In [21]:
df = convert_column_types(df, columns_types=expected_formats)


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Player Name                   48000 non-null  string        
 1   Team                          48000 non-null  string        
 2   World                         48000 non-null  string        
 3   Vehicle Type                  48000 non-null  string        
 4   Companion                     48000 non-null  string        
 5   Kart Racing Rank              48000 non-null  string        
 6   Platforming Rank              48000 non-null  string        
 7   Boss Battle Rank              48000 non-null  string        
 8   Power-Ups Used                48000 non-null  int32         
 9   Kart Role                     48000 non-null  string        
 10  Team Points                   48000 non-null  int32         
 11  Lives Lost                  

In [23]:
numeric_columns = [
    'Power-Ups Used', 'Team Points', 'Lives Lost',
    'Coins Spent in Toad Town', 'Levels Completed', 'Times Hit by Enemies'
]

df[numeric_columns].isnull().sum()

Power-Ups Used              0
Team Points                 0
Lives Lost                  0
Coins Spent in Toad Town    0
Levels Completed            0
Times Hit by Enemies        0
dtype: int64

In [24]:
numeric_columns = [
    'Power-Ups Used', 'Team Points', 'Lives Lost',
    'Coins Spent in Toad Town', 'Levels Completed', 'Times Hit by Enemies'
]

df[numeric_columns] = df[numeric_columns].fillna(0).astype('int32')

print("Step 4B complete — missing numeric values filled and converted to int32")
df[numeric_columns].info()

Step 4B complete — missing numeric values filled and converted to int32
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Power-Ups Used            48000 non-null  int32
 1   Team Points               48000 non-null  int32
 2   Lives Lost                48000 non-null  int32
 3   Coins Spent in Toad Town  48000 non-null  int32
 4   Levels Completed          48000 non-null  int32
 5   Times Hit by Enemies      48000 non-null  int32
dtypes: int32(6)
memory usage: 1.1 MB


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Player Name                   48000 non-null  string        
 1   Team                          48000 non-null  string        
 2   World                         48000 non-null  string        
 3   Vehicle Type                  48000 non-null  string        
 4   Companion                     48000 non-null  string        
 5   Kart Racing Rank              48000 non-null  string        
 6   Platforming Rank              48000 non-null  string        
 7   Boss Battle Rank              48000 non-null  string        
 8   Power-Ups Used                48000 non-null  int32         
 9   Kart Role                     48000 non-null  string        
 10  Team Points                   48000 non-null  int32         
 11  Lives Lost                  

In [26]:
target_name = "Mario"
similarity_threshold = 0.8

similar_names = []

for name in df['Player Name'].dropna().unique():
    similarity = check_similarity(name.lower(), target_name.lower())
    if similarity >= similarity_threshold and name.lower() != target_name.lower():
        similar_names.append((name, similarity))

mario_variants = pd.DataFrame(similar_names, columns=['Possible Variant', 'Similarity'])
display(mario_variants)
print(f"Step 5 complete — found {len(mario_variants)} possible variations of 'Mario' (≥ 80%).")

,Possible Variant,Similarity
0,MZario,0.833333
1,Mariom,1.000000
2,MariAo,1.000000
3,Myario,0.833333
4,MariQo,0.833333
...,...,...
93,MaWrio,0.833333
94,MarioV,0.833333
95,Marxio,0.833333
96,MarioS,0.833333


Step 5 complete — found 98 possible variations of 'Mario' (≥ 80%).


In [27]:
valid_names = [
    'Mario', 'Luigi', 'Peach', 'Daisy', 'Yoshi', 'Toad', 'Toadette',
    'Rosalina', 'Wario', 'Waluigi', 'Bowser', 'Bowser Jr.'
]

corrections = {}

for name in df['Player Name'].dropna().unique():
    best_match = max(valid_names, key=lambda x: check_similarity(name.lower(), x.lower()))
    similarity = check_similarity(name.lower(), best_match.lower())
    if similarity >= 0.8 and name != best_match:
        corrections[name] = best_match

df['Player Name'] = df['Player Name'].replace(corrections)

print('Step 6 complete — standardized Player Name automatically.')
print('Unique Player Names now:')
print(sorted(df['Player Name'].unique()))

Step 6 complete — standardized Player Name automatically.
Unique Player Names now:
['Bowser', 'Bowser Jr.', 'Daisy', 'Luigi', 'Mario', 'Peach', 'Rosalina', 'Toad', 'Toadette', 'Unknown Player', 'Waluigi', 'Wario', 'Yoshi']


### Clean and Standardize Text Fields
- Strip extra spaces from `'Vehicle Type'`, `'World'`, and `'Primary Game'`.
- Apply Titlecase to `'Team'`, `'Companion'`, and `'World'` for uniformity and readability.


In [28]:
columns_to_strip = ['Vehicle Type', 'World', 'Primary Game']

for col in columns_to_strip:
    if df[col].dtype == 'string' or df[col].dtype == 'object':
        df[col] = df[col].str.strip()

print('Step 7 complete — removed leading and trailing spaces from:', columns_to_strip)

Step 7 complete — removed leading and trailing spaces from: ['Vehicle Type', 'World', 'Primary Game']


In [29]:
df[columns_to_strip].head(10)

,Vehicle Type,World,Primary Game
0,Comet Bike,Yoshi's Island,Mario Tennis Aces
1,Circuit Special,Donut Plains,Mario Tennis Aces
2,Biddybuggy,Yoshi's Island,Mario Kart 8 Deluxe
3,Pipe Frame,Star World,Super Mario Bros.
4,Pipe Frame,Mushroom Kingdom,Super Mario World
5,Comet Bike,Toad Town,Super Mario World
6,Comet Bike,Star World,Mario Party Superstars
7,Scooter,Star World,Mario Strikers
8,Comet Bike,Star World,Super Mario Odyssey
9,Standard Kart,Mushroom Kingdom,Super Mario World


In [30]:
import string

titlecase_columns = ['Team', 'Companion', 'World']

for col in titlecase_columns:
    if df[col].dtype == 'string' or df[col].dtype == 'object':
        df[col] = df[col].astype('string').str.strip().apply(
            lambda value: string.capwords(value.lower()) if pd.notna(value) else value
        ).astype('string')

print('Step 8 complete — applied title casing to:', titlecase_columns)


Step 8 complete — applied title casing to: ['Team', 'Companion', 'World']


In [31]:
df[titlecase_columns].head(10)

,Team,Companion,World
0,Toad Brigade,Polterpup,Yoshi's Island
1,Green Caps,Koopa Troopa,Donut Plains
2,No Team,Goomba,Yoshi's Island
3,Toad Brigade,Goomba,Star World
4,Koopa Clan,Toad,Mushroom Kingdom
5,Red Caps,Goomba,Toad Town
6,Green Caps,Goomba,Star World
7,Green Caps,Luma,Star World
8,Toad Brigade,Polterpup,Star World
9,Tricksters,Polterpup,Mushroom Kingdom


### Silver Layer Complete
All data cleaning, validation, and standardization steps are finished.
The resulting dataset (`silver_transactions.parquet`) is schema-consistent, free of critical nulls, and ready for Gold-layer analysis.

In [32]:
import os

silver_output_path = './Medallion Architecture/silver/silver_transactions.parquet'
os.makedirs(os.path.dirname(silver_output_path), exist_ok=True)

df.to_parquet(silver_output_path, index=False)


In [33]:
pd.read_parquet('./Medallion Architecture/silver/silver_transactions.parquet').info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Player Name                   48000 non-null  string        
 1   Team                          48000 non-null  string        
 2   World                         48000 non-null  string        
 3   Vehicle Type                  48000 non-null  string        
 4   Companion                     48000 non-null  string        
 5   Kart Racing Rank              48000 non-null  string        
 6   Platforming Rank              48000 non-null  string        
 7   Boss Battle Rank              48000 non-null  string        
 8   Power-Ups Used                48000 non-null  int32         
 9   Kart Role                     48000 non-null  string        
 10  Team Points                   48000 non-null  int32         
 11  Lives Lost                  